# SK-Ana Python: synthetic spectrokinetic example

All observations below are generated in memory; no experimental or patient data are used. Install with `pip install -e .[dev]` from `python/` before running.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skana import MatrixData, als, export_matrix, read_matrix, svd
from skana.plots import diagnostics, image, profiles

rng = np.random.default_rng(7)
delay = np.linspace(0, 8, 32)
wavelength = np.linspace(400, 700, 60)
c_true = np.column_stack((np.exp(-delay / 2), 1 - np.exp(-delay / 2)))
s_true = np.column_stack((np.exp(-0.5 * ((wavelength - 470) / 26)**2),
                          0.8 * np.exp(-0.5 * ((wavelength - 610) / 37)**2)))
matrix = c_true @ s_true.T + 0.002 * rng.normal(size=(delay.size, wavelength.size))
data = MatrixData(matrix, delay, wavelength, 'synthetic')
loaded = read_matrix(export_matrix(data), preset='CSV')
assert np.allclose(loaded.matrix, data.matrix)
print('Loaded shape:', loaded.matrix.shape)
display(image(loaded, loaded.matrix, 'Synthetic data'))

## SVD and rank-two reconstruction

In [ ]:
decomp = svd(loaded, rank=2)
display(decomp.statistics)
display(profiles(loaded, decomp.u[:, :2], decomp.vt[:2, :].T, 'SVD'))
display(diagnostics(loaded, decomp.model, decomp.residual, 'SVD rank 2'))

## MCR-ALS, reconstruction and residual diagnostics

In [ ]:
fit = als(loaded, n_components=2, maxiter=300, thresh=1e-7)
print(f'LOF = {fit.lof:.4f}%, iterations = {fit.iter}, converged = {fit.converged}')
assert fit.model.shape == loaded.matrix.shape
assert np.allclose(fit.resid, loaded.matrix - fit.model)
display(profiles(loaded, fit.c, fit.s, 'MCR-ALS'))
display(diagnostics(loaded, fit.model, fit.resid, 'MCR-ALS'))
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(fit.resid.ravel(), bins=30)
axes[0].set(xlabel='Residual', ylabel='Count')
axes[1].semilogy(np.linalg.svd(fit.resid, compute_uv=False), 'o-')
axes[1].set(xlabel='Index', ylabel='Residual singular value')
fig.tight_layout()
display(fig)
plt.close(fig)

## Mechanistic A → B model (Python only)

A mass-action ODE generates concentrations on the physical time axis. Variable-projection spectra and a local MAP fit recover the rate; the residual checks the reconstruction.

In [ ]:
from skana import concentrations, fit_kinetics, parse_scheme

scheme = parse_scheme("A -> B ; 0.5 / 1.3\nc0_A_1 = 1\neps_A = 1\neps_B = 0.8\n")
blank = MatrixData(np.zeros_like(matrix), delay, wavelength, 'kinetic_synthetic')
c_model = concentrations(blank, scheme)
s_model = np.column_stack((np.exp(-((wavelength - 470) / 36)**2),
                           np.exp(-((wavelength - 605) / 42)**2)))
s_model /= s_model.max(axis=0)
s_model[:, 1] *= 0.8
kinetic_data = MatrixData(c_model @ s_model.T + 0.0005 * rng.normal(size=matrix.shape),
                          delay, wavelength, 'kinetic_synthetic')
kinetic_fit = fit_kinetics(kinetic_data, scheme)
print('Fitted rate:', kinetic_fit.parameters['k_1'])
print('Kinetic LOF:', kinetic_fit.lof)
assert kinetic_fit.model.shape == kinetic_data.matrix.shape
assert np.allclose(kinetic_fit.model + kinetic_fit.residual, kinetic_data.matrix)
display(profiles(kinetic_data, kinetic_fit.c, kinetic_fit.s, 'Mechanistic fit'))
display(diagnostics(kinetic_data, kinetic_fit.model, kinetic_fit.residual,
                    'Mechanistic fit'))